# character identification

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from sklearn.model_selection import train_test_split
import torch

# Load annotated dataset
df = pd.read_csv("Data Inggris - NER_american.csv")
df['word'] = df['word'].astype(str)
# Only keep relevant columns
df = df[['storyID', 'sentenceID', 'word', 'BIO_tag']]

In [ ]:
# 2. Buat label2id dan id2label dari keseluruhan file
unique_labels = df['BIO_tag'].unique().tolist()
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {v: k for k, v in label2id.items()}

# 3. Fungsi bantu untuk memproses dataframe jadi list of sentences & label
def prepare_data(data_df):
    grouped = data_df.groupby(['storyID', 'sentenceID'])
    sentences = []
    labels = []
    story_ids = []

    for (story_id, _), group in grouped:
        word_list = group['word'].tolist()
        label_list = group['BIO_tag'].map(label2id).tolist()
        sentences.append(word_list)
        labels.append(label_list)
        story_ids.append(story_id)

    return {
        'tokens': sentences,
        'ner_tags': labels,
        'story_id': story_ids
    }

# 4. Membagi 1 dataset menjadi Train (80%) dan Test (20%) berdasarkan StoryID
# Kita menggunakan train_test_split yang sudah Anda import sebelumnya
unique_stories = df['storyID'].unique()
train_stories, test_stories = train_test_split(unique_stories, test_size=0.2, random_state=42)

df_train = df[df['storyID'].isin(train_stories)]
df_test = df[df['storyID'].isin(test_stories)]

# 5. Eksekusi fungsi prepare_data
train_data = prepare_data(df_train)
test_data = prepare_data(df_test)

# 6. Ubah ke HuggingFace Dataset dan gabungkan ke DatasetDict
train_dataset = Dataset.from_dict(train_data)
test_dataset = Dataset.from_dict(test_data)

dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

# 7. Load Tokenizer
model_checkpoint = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding='max_length',
        max_length=128
    )

    all_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Token pertama dari suatu kata
                label_ids.append(labels[word_idx])
            else:
                # Subword dari kata yang sama → konversi B-XXX → I-XXX
                original_label = labels[word_idx]
                original_label_name = id2label[original_label]

                if original_label_name.startswith("B-"):
                    i_label_name = "I-" + original_label_name[2:]
                    i_label_id = label2id.get(i_label_name, original_label)
                    label_ids.append(i_label_id)
                else:
                    label_ids.append(original_label)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

In [ ]:
# Apply preprocessing
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# Load model
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)


Map:   0%|          | 0/10064 [00:00<?, ? examples/s]

Map:   0%|          | 0/2384 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |                                                                                     
-------------------------+------------+-------------------------------------------------------------------------------------
bert.pooler.dense.bias   | UNEXPECTED |                                                                                     
bert.pooler.dense.weight | UNEXPECTED |                                                                                     
classifier.weight        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9, 768]) vs model:torch.Size([3, 768])
classifier.bias          | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([9]) vs model:torch.Size([3])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH	:ckpt weights were loaded, but they did not match the

In [ ]:
pip install seqeval

In [ ]:
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_labels = []
    pred_labels = []

    for true, pred in zip(labels, predictions):
        true_seq = []
        pred_seq = []
        for t, p in zip(true, pred):
            if t != -100:
                true_seq.append(id2label[t])
                pred_seq.append(id2label[p])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq)

    return {
        "accuracy": accuracy_score(true_labels, pred_labels),
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.036670,0.032065,0.991830,0.851475,0.827893,0.839519
2,0.006716,0.042942,0.991555,0.860759,0.807122,0.833078
3,0.002715,0.051281,0.992463,0.858006,0.842730,0.850299
4,0.000677,0.053998,0.992083,0.877814,0.810089,0.842593
5,0.000305,0.059749,0.992294,0.874089,0.830861,0.851927
6,0.000303,0.061332,0.992252,0.871503,0.831850,0.851215
7,0.000154,0.063330,0.992484,0.883598,0.825915,0.853783
8,0.000067,0.069460,0.992442,0.889722,0.821958,0.854499
9,0.000026,0.071559,0.992400,0.878125,0.833828,0.855403
10,0.000052,0.072002,0.992526,0.882845,0.834817,0.858160


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=6290, training_loss=0.003911583605762837, metrics={'train_runtime': 268.6754, 'train_samples_per_second': 374.578, 'train_steps_per_second': 23.411, 'total_flos': 6574285836656640.0, 'train_loss': 0.003911583605762837, 'epoch': 10.0})

In [ ]:
from transformers import AutoModelForTokenClassification

# Contoh: simpan model dan tokenizer (setelah training)
model.save_pretrained("saved_model/my_model")
tokenizer.save_pretrained("saved_model/my_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_model/my_model/tokenizer_config.json',
 'saved_model/my_model/tokenizer.json')

In [ ]:
import shutil
from google.colab import files

# Zip
shutil.make_archive("my_model", 'zip', "saved_model/my_model")

# Unduh
files.download("my_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import TokenClassificationPipeline
from collections import defaultdict
import torch

# Setup pipeline untuk NER
ner_pipe = TokenClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # Gabungkan B- dan I- ke satu entity_group
    device=0 if torch.cuda.is_available() else -1
)

# Bersihkan token dari subword marker
def clean_word(word):
    return word.replace("##", "").strip()

# Simpan hasil karakter berdasarkan story_id
storywise_characters = defaultdict(set)

# Proses setiap kalimat dari dataset test
for story_id, tokens in zip(dataset["test"]["story_id"], dataset["test"]["tokens"]):
    sentence = " ".join(tokens)
    preds = ner_pipe(sentence)

    entity_buffer = []
    prev_entity_type = None

    for pred in preds:
        label = pred["entity_group"]  # misalnya: PNAME, OBJ, ANM
        word = clean_word(pred["word"])

        if label in ["PER"]:  # sesuaikan dengan label karakter
            if entity_buffer:
                name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
                if name:
                    storywise_characters[story_id].add(name)
            entity_buffer = [word]
        else:
            if entity_buffer:
                name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
                if name:
                    storywise_characters[story_id].add(name)
                entity_buffer = []

        prev_entity_type = label

    # Flush terakhir
    if entity_buffer:
        name = tokenizer.convert_tokens_to_string(entity_buffer).strip().title()
        if name:
            storywise_characters[story_id].add(name)

# ✅ Tampilkan hasil akhir
for story, chars in storywise_characters.items():
    print(f"Story ID: {story}\nCharacters: {sorted(chars)}\n")


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Story ID: 10
Characters: ['Councillor', 'Daughter', 'King', 'Man', 'Princess', 'Queen']

Story ID: 16
Characters: ['Baby', 'Dragons', 'Mother', 'Prince']

Story ID: 17
Characters: ['Brother', 'James', 'Man', 'Mark Twain', 'Men', 'Mr Dreiser']

Story ID: 19
Characters: ['Blacksmith', 'Brother', 'Child', 'Father', 'Girl', 'Horses', 'Man', 'Tom', 'Wife']

Story ID: 31
Characters: ['Abbot', 'Friends', 'Horse', 'John', 'King', 'King John', 'Man', 'Master', 'Men', 'Sir Abbot']

Story ID: 46
Characters: ['Car - Thage', 'Children', 'Man', 'Men', 'Reg', 'Wife']

Story ID: 56
Characters: ['Doctor', 'Dr Frayley', 'Dr Mannering', 'Friend', 'Friends', 'Ghost', 'Ghosts', 'Man', 'Men', 'Servant', 'Student']

Story ID: 57
Characters: ['Gentlemen', 'Henry Saylor', 'Ladies', 'Man', 'Men', 'Woman', 'Women']

Story ID: 61
Characters: ['Andrus', 'Andrus C Palmer', 'John Holcomb', 'Lawyer', 'Man', 'Men', 'Philip Eckert', 'Teacher']

Story ID: 66
Characters: ['Children', 'Dog', 'Dogs', 'Father', 'Lady', 'Man

In [ ]:
import pandas as pd

# Assuming this mapping exists from earlier
story_id_to_title = df_test.groupby('storyID').first().to_dict()

# Flatten the results into one row per character
char_data_flat = {
    "story_id": [],
    "character": []
}

for story_id, characters in storywise_characters.items():
    for character in sorted(characters):
        char_data_flat["story_id"].append(story_id)
        char_data_flat["character"].append(character)

# Convert to DataFrame
df_flat = pd.DataFrame(char_data_flat)

# Show it
df_flat


,story_id,character
0,10,Councillor
1,10,Daughter
2,10,King
3,10,Man
4,10,Princess
...,...,...
401,183,Mother
402,183,Will
403,183,William Spencer
404,187,Ladies
